# Flight data preprocessing and aircraft-pair construction

This notebook prepares January-April 2026 BTS on-time data for Southwest (WN), profiles the raw fields, converts local schedule and actual times to UTC, links consecutive flights by aircraft tail, and writes processed datasets and reports.

Run all cells from top to bottom after restarting the kernel. The raw source files are not modified. Install the packages listed in `requirements.txt` first.

**Scope note:** this notebook creates a cleaned analysis dataset and sequence pairs. It does not yet fit model-specific imputers, encoders, scalers, feature selectors, or a predictive model; those belong in the later modeling workflow after the train/test split. Binning is also deferred until the analysis shows that ranges add value. These choices depend on the final prediction point and model, so applying them here would risk leakage or unnecessary transformations.

In [1]:
"""Reusable helpers and configuration for the preprocessing notebook."""

from __future__ import annotations

import json
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd


# Locate the project root from the notebook's current working directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "reference" / "airport_timezones.csv").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data" / "reference" / "airport_timezones.csv").is_file():
    raise FileNotFoundError("Open this notebook from the flight project folder.")
RAW_PATTERN = "data/raw/T_ONTIME_REPORTING_*/T_ONTIME_REPORTING.csv"
TIMEZONE_FILE = ROOT / "data" / "reference" / "airport_timezones.csv"
OUTPUT_DIR = ROOT / "data" / "processed"
REPORT_DIR = ROOT / "reports" / "metrics"
ANALYSIS_DIR = ROOT / "reports" / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
CARRIER = "WN"
MONTH_LABEL = "2026-01_to_2026-04"
MAX_ARRIVAL_HOURS = 30
MAX_TURNAROUND_HOURS = 24

REQUIRED_COLUMNS = {
    "YEAR",
    "MONTH",
    "DAY_OF_MONTH",
    "OP_UNIQUE_CARRIER",
    "TAIL_NUM",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",
    "CRS_ARR_TIME",
    "ARR_TIME",
    "ARR_DELAY_NEW",
    "CANCELLED",
    "DIVERTED",
}

SOURCE_COLUMNS = [
    "YEAR",
    "MONTH",
    "DAY_OF_MONTH",
    "OP_UNIQUE_CARRIER",
    "TAIL_NUM",
    "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID",
    "ORIGIN",
    "DEST_AIRPORT_ID",
    "DEST",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",
    "DEP_DELAY_NEW",
    "CRS_ARR_TIME",
    "ARR_TIME",
    "ARR_DELAY_NEW",
    "ARR_DEL15",
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED",
]


def parse_clock(value: object, local_date: date) -> datetime | None:
    """Parse a BTS local HHMM clock field, including 2400 as next-day midnight."""
    if pd.isna(value):
        return None
    raw = str(value).strip()
    if not raw:
        return None
    try:
        numeric = int(float(raw))
    except (TypeError, ValueError, OverflowError):
        return None
    if numeric < 0 or numeric > 2400:
        return None
    hhmm = f"{numeric:04d}"
    hour, minute = int(hhmm[:2]), int(hhmm[2:])
    if minute > 59:
        return None
    if hour == 24 and minute == 0:
        return datetime.combine(local_date + timedelta(days=1), time.min)
    if hour > 23:
        return None
    return datetime.combine(local_date, time(hour, minute))


def to_utc(local_dt: datetime | None, tz_name: str | None) -> datetime | None:
    """Convert a local wall time to UTC; shift nonexistent DST times forward."""
    if local_dt is None or not tz_name:
        return None
    zone = ZoneInfo(tz_name)
    candidate = local_dt.replace(tzinfo=zone, fold=0)
    round_trip = candidate.astimezone(timezone.utc).astimezone(zone)
    if round_trip.replace(tzinfo=None) == local_dt:
        return candidate.astimezone(timezone.utc)
    # A spring-forward clock time does not exist. ZoneInfo's fold=0 round trip
    # moves it forward by the DST gap, matching pandas' shift_forward behavior.
    if round_trip.replace(tzinfo=None) > local_dt:
        return round_trip.astimezone(timezone.utc)
    # For an ambiguous fall-back time, fold=0 consistently chooses the earlier
    # occurrence. This study period is Jan-Mar, so fall-back is outside the sample.
    return candidate.astimezone(timezone.utc)


def arrival_to_utc(
    clock_value: object,
    departure_local_date: date,
    tz_name: str | None,
    departure_utc: datetime | None,
) -> datetime | None:
    """Resolve an arrival clock to the first plausible date on/after departure."""
    if departure_utc is None:
        return None
    for day_offset in range(0, 2):
        local_dt = parse_clock(clock_value, departure_local_date + timedelta(days=day_offset))
        arrival_utc = to_utc(local_dt, tz_name)
        if arrival_utc is None:
            continue
        elapsed = (arrival_utc - departure_utc).total_seconds() / 3600
        if 0 <= elapsed <= MAX_ARRIVAL_HOURS:
            return arrival_utc
    return None


def dt_series(values: list[datetime | None]) -> pd.Series:
    return pd.to_datetime(pd.Series(values), utc=True)




## 1. Profile and normalize the raw records

The notebook first reports shape, column types, numerical/categorical fields, descriptive statistics, and missing counts/percentages for all columns in the WN subset of each monthly source file. It then narrows to fields needed for flight identity, route, schedule/actual times, delay outcome, cancellation, and diversion. This focused working table reduces memory and keeps the transformation scope clear.

Local clock fields are converted using the origin/destination IANA time zones so flights across time zones can be ordered consistently. `2400` and next-day arrivals are handled explicitly. Unresolved actual times are retained as missing in the flight-level output; they are not imputed because for cancellations and incomplete operations, absence has operational meaning.

**Missing-value decisions:** missing actual departure/arrival times and delay values are retained as missing because they can indicate cancellation, diversion, or incomplete reporting; they are not fabricated by imputation. Such rows remain in flight-level/cancellation analyses but are excluded from operated-flight pairs when required timestamps or delays are unavailable. Missing tail numbers remain unknown and are excluded from aircraft-chain linking. Blank cancellation reasons are labeled `Unknown / blank`. The profile reports the exact counts and rates per month and field.

BTS downloads can differ in optional fields when the selected column set changes. The essential fields are validated; missing precomputed delay flags are not required because the departure target is derived from `DepDelay`.

In [2]:
raw_files = sorted(Path(ROOT).glob(RAW_PATTERN))
if len(raw_files) != 4:
    raise SystemExit(f"Expected 4 extracted monthly BTS CSVs, found {len(raw_files)}.")
if not TIMEZONE_FILE.is_file():
    raise SystemExit(f"Missing airport timezone lookup: {TIMEZONE_FILE}")

timezone_table = pd.read_csv(TIMEZONE_FILE, dtype={"airport_code": "string"})
timezone_map = dict(zip(timezone_table.airport_code, timezone_table.timezone))

frames: list[pd.DataFrame] = []
file_summary: list[dict[str, object]] = []
raw_column_profiles: list[pd.DataFrame] = []
raw_descriptive_statistics: list[pd.DataFrame] = []
raw_file_shapes: list[dict[str, object]] = []
for path in raw_files:
    frame = pd.read_csv(path, low_memory=False)
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise SystemExit(f"{path.name} is missing required fields: {missing}")
    total_source_rows = len(frame)
    frame = frame.loc[frame["OP_UNIQUE_CARRIER"].eq(CARRIER)].copy().reset_index(drop=True)

    # Profile every available raw field for the selected carrier before narrowing
    # the working table to project-relevant fields.
    raw_file_shapes.append({
        "file": str(path.relative_to(ROOT)),
        "source_rows_all_carriers": int(total_source_rows),
        "rows_for_WN": int(len(frame)),
        "columns": int(frame.shape[1]),
    })
    raw_column_profiles.append(pd.DataFrame({
        "FILE": str(path.relative_to(ROOT)),
        "COLUMN": frame.columns,
        "DTYPE": frame.dtypes.astype(str).values,
        "MISSING_COUNT": frame.isna().sum().values,
        "MISSING_PERCENT": (frame.isna().mean().values * 100).round(3),
        "COLUMN_TYPE": ["numeric" if pd.api.types.is_numeric_dtype(t) else "categorical_or_text" for t in frame.dtypes],
    }))
    raw_descriptive_statistics.append(
        frame.describe(include="all").transpose().assign(FILE=str(path.relative_to(ROOT))).reset_index(names="COLUMN")
    )
    frame = frame[[column for column in SOURCE_COLUMNS if column in frame.columns]].copy()
    # BTS may omit optional precomputed delay fields in a selected download.
    # The project derives DEP_DEL15 from DEP_DELAY and does not require these.
    for optional_column in ["DEP_DELAY_NEW", "ARR_DEL15"]:
        if optional_column not in frame.columns:
            frame[optional_column] = pd.NA
    frame["FLIGHT_DATE"] = pd.to_datetime(
        frame[["YEAR", "MONTH", "DAY_OF_MONTH"]].astype("int64").rename(
            columns={"YEAR": "year", "MONTH": "month", "DAY_OF_MONTH": "day"}
        ),
        errors="coerce",
    )
    if frame["FLIGHT_DATE"].isna().any():
        raise ValueError(f"Could not parse flight dates in {path.name}.")
    frame["ORIGIN_TZ"] = frame["ORIGIN"].map(timezone_map)
    frame["DEST_TZ"] = frame["DEST"].map(timezone_map)
    missing_airports = sorted(
        set(frame.loc[frame["ORIGIN_TZ"].isna(), "ORIGIN"].dropna().astype(str))
        | set(frame.loc[frame["DEST_TZ"].isna(), "DEST"].dropna().astype(str))
    )
    if missing_airports:
        raise ValueError(f"No timezone mapping for airport codes: {missing_airports}")

    dates = [x.date() for x in frame["FLIGHT_DATE"]]
    sched_dep_local = [parse_clock(v, d) for v, d in zip(frame["CRS_DEP_TIME"], dates)]
    actual_dep_local = [parse_clock(v, d) for v, d in zip(frame["DEP_TIME"], dates)]
    sched_dep_utc = [
        to_utc(local, tz) for local, tz in zip(sched_dep_local, frame["ORIGIN_TZ"])
    ]
    actual_dep_utc = [
        to_utc(local, tz) for local, tz in zip(actual_dep_local, frame["ORIGIN_TZ"])
    ]
    sched_arr_utc = [
        arrival_to_utc(clock, local.date() if local else day, tz, dep_utc)
        for clock, local, day, tz, dep_utc in zip(
            frame["CRS_ARR_TIME"], sched_dep_local, dates, frame["DEST_TZ"], sched_dep_utc
        )
    ]
    actual_arr_utc = [
        arrival_to_utc(clock, local.date() if local else day, tz, dep_utc)
        for clock, local, day, tz, dep_utc in zip(
            frame["ARR_TIME"], actual_dep_local, dates, frame["DEST_TZ"], actual_dep_utc
        )
    ]

    frame["SCHEDULED_DEP_UTC"] = dt_series(sched_dep_utc)
    frame["ACTUAL_DEP_UTC"] = dt_series(actual_dep_utc)
    frame["SCHEDULED_ARR_UTC"] = dt_series(sched_arr_utc)
    frame["ACTUAL_ARR_UTC"] = dt_series(actual_arr_utc)
    frame["DEP_DELAY"] = pd.to_numeric(frame["DEP_DELAY"], errors="coerce")
    frame["DEP_DELAY_NEW"] = pd.to_numeric(frame["DEP_DELAY_NEW"], errors="coerce")
    frame["ARR_DELAY_NEW"] = pd.to_numeric(frame["ARR_DELAY_NEW"], errors="coerce")
    frame["CANCELLED"] = pd.to_numeric(frame["CANCELLED"], errors="coerce").fillna(0).astype(int)
    frame["DIVERTED"] = pd.to_numeric(frame["DIVERTED"], errors="coerce").fillna(0).astype(int)
    frame["DEP_DEL15"] = (frame["DEP_DELAY"] >= 15).astype("int8")
    frame["DEPARTURE_HOUR_LOCAL"] = pd.to_numeric(
        frame["CRS_DEP_TIME"], errors="coerce"
    ).floordiv(100) % 24
    frame["DAY_OF_WEEK"] = frame["FLIGHT_DATE"].dt.dayofweek
    file_summary.append(
        {
            "file": str(path.relative_to(ROOT)),
            "rows_for_WN": int(len(frame)),
            "cancelled_for_WN": int(frame["CANCELLED"].sum()),
            "diverted_for_WN": int(frame["DIVERTED"].sum()),
            "missing_tail_for_WN": int(frame["TAIL_NUM"].isna().sum()),
            "unparsed_actual_departures": int(frame["ACTUAL_DEP_UTC"].isna().sum()),
            "unparsed_actual_arrivals": int(frame["ACTUAL_ARR_UTC"].isna().sum()),
        }
    )
    frames.append(frame)

# Persist the exploratory profile for review and the project report.
raw_column_profile = pd.concat(raw_column_profiles, ignore_index=True)
raw_statistics = pd.concat(raw_descriptive_statistics, ignore_index=True)
raw_profile_path = REPORT_DIR / "day2_raw_column_profile.csv"
raw_statistics_path = REPORT_DIR / "day2_raw_descriptive_statistics.csv"
raw_profile_summary_path = REPORT_DIR / "day2_raw_dataset_profile.json"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
raw_column_profile.to_csv(raw_profile_path, index=False)
raw_statistics.to_csv(raw_statistics_path, index=False)
first_month_profile = raw_column_profile.loc[raw_column_profile["FILE"].eq(raw_column_profile["FILE"].iloc[0])]
raw_profile_summary = {
    "scope": "Southwest (WN) rows in each BTS monthly source file",
    "files": raw_file_shapes,
    "column_count": int(first_month_profile.shape[0]),
    "numeric_columns": first_month_profile.loc[first_month_profile["COLUMN_TYPE"].eq("numeric"), "COLUMN"].tolist(),
    "categorical_or_text_columns": first_month_profile.loc[first_month_profile["COLUMN_TYPE"].eq("categorical_or_text"), "COLUMN"].tolist(),
    "outputs": [str(raw_profile_path.relative_to(ROOT)), str(raw_statistics_path.relative_to(ROOT))],
}
raw_profile_summary_path.write_text(json.dumps(raw_profile_summary, indent=2), encoding="utf-8")


1453

### Review raw-data quality

These tables expose the source shape, column types, numeric/categorical classification, missing-value counts and percentages, and descriptive statistics. The CSV reports preserve full detail for all four months; the notebook displays the combined missingness/type profile and descriptive statistics for the first month to keep the output reviewable.

In [3]:
profile_review = raw_column_profile.groupby("COLUMN", as_index=False).agg(
    DTYPE=("DTYPE", "first"),
    COLUMN_TYPE=("COLUMN_TYPE", "first"),
    MISSING_COUNT=("MISSING_COUNT", "sum"),
)
profile_review["MISSING_PERCENT"] = (
    profile_review["MISSING_COUNT"] / sum(item["rows_for_WN"] for item in raw_file_shapes) * 100
).round(3)
print("Source shapes (one row per monthly BTS file):")
display(pd.DataFrame(raw_file_shapes))
print("Column types and missingness across all WN rows:")
display(profile_review)
first_source_name = raw_file_shapes[0]["file"]
print(f"Descriptive statistics for {first_source_name}; complete per-month results are in the saved CSV:")
display(raw_statistics.loc[raw_statistics["FILE"].eq(first_source_name)].reset_index(drop=True))

Source shapes (one row per monthly BTS file):


,file,source_rows_all_carriers,rows_for_WN,columns
0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...,544003,104002,23
1,T_ONTIME_REPORTING_20261027_145800\T_ONTIME_RE...,515037,100783,23
2,T_ONTIME_REPORTING_20261027_150025\T_ONTIME_RE...,612102,122298,23
3,T_ONTIME_REPORTING_20261029_080530\T_ONTIME_RE...,597919,117814,17


Column types and missingness across all WN rows:


,COLUMN,DTYPE,COLUMN_TYPE,MISSING_COUNT,MISSING_PERCENT
0,ARR_DEL15,float64,numeric,6283,1.412
1,ARR_DELAY_NEW,float64,numeric,6767,1.521
2,ARR_TIME,float64,numeric,6093,1.370
3,CANCELLATION_CODE,object,categorical_or_text,438928,98.658
4,CANCELLED,float64,numeric,0,0.000
5,CRS_ARR_TIME,int64,numeric,0,0.000
6,CRS_DEP_TIME,int64,numeric,0,0.000
7,DAY_OF_MONTH,int64,numeric,0,0.000
8,DEP_DELAY,float64,numeric,5859,1.317
9,DEP_DELAY_NEW,float64,numeric,5567,1.251


Descriptive statistics for T_ONTIME_REPORTING_20261027_145650\T_ONTIME_REPORTING.csv; complete per-month results are in the saved CSV:


,COLUMN,count,unique,top,freq,mean,std,min,25%,50%,75%,max,FILE
0,YEAR,104002.0,NaN,NaN,NaN,2026.0,0.0,2026.0,2026.0,2026.0,2026.0,2026.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
1,MONTH,104002.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
2,DAY_OF_MONTH,104002.0,NaN,NaN,NaN,15.501144,9.019414,1.0,7.0,15.0,23.0,31.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
3,OP_UNIQUE_CARRIER,104002,1,WN,104002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
4,TAIL_NUM,103490,795,N8944Q,159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
5,OP_CARRIER_FL_NUM,104002.0,NaN,NaN,NaN,2226.9765,1315.293705,1.0,1165.0,2149.0,3118.0,5596.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
6,ORIGIN_AIRPORT_ID,104002.0,NaN,NaN,NaN,12748.36707,1534.190927,10140.0,11259.0,12892.0,14107.0,15624.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
7,ORIGIN,104002,104,LAS,6704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
8,DEST_AIRPORT_ID,104002.0,NaN,NaN,NaN,12748.662237,1534.102482,10140.0,11259.0,12892.0,14107.0,15624.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...
9,DEST_AIRPORT_SEQ_ID,104002.0,NaN,NaN,NaN,1274870.393146,153410.05283,1014006.0,1125905.0,1289209.0,1410703.0,1562405.0,T_ONTIME_REPORTING_20261027_145650\T_ONTIME_RE...


## 2. Combine and export flight records

The four monthly WN subsets are combined and sorted by tail and actual departure time. Blank tail identifiers remain missing. Cancellations and diversions stay in the flight-level export so they can be analyzed as outcomes, not silently discarded.

In [4]:
flights = pd.concat(frames, ignore_index=True)
flights["TAIL_NUM"] = flights["TAIL_NUM"].astype("string").str.strip()
flights.loc[flights["TAIL_NUM"].isin(["", "nan", "None"]), "TAIL_NUM"] = pd.NA
flights = flights.sort_values(
    ["TAIL_NUM", "ACTUAL_DEP_UTC", "FLIGHT_DATE", "OP_CARRIER_FL_NUM"],
    kind="mergesort",
    na_position="last",
).reset_index(drop=True)

all_flights_path = OUTPUT_DIR / f"wn_flights_{MONTH_LABEL}.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_columns = [
    "YEAR", "MONTH", "DAY_OF_MONTH", "FLIGHT_DATE", "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM", "TAIL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME", "DEP_TIME",
    "DEP_DELAY", "DEP_DELAY_NEW", "DEP_DEL15", "CRS_ARR_TIME", "ARR_TIME",
    "ARR_DELAY_NEW", "ARR_DEL15", "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
    "ORIGIN_TZ", "DEST_TZ", "SCHEDULED_DEP_UTC", "ACTUAL_DEP_UTC",
    "SCHEDULED_ARR_UTC", "ACTUAL_ARR_UTC", "DEPARTURE_HOUR_LOCAL", "DAY_OF_WEEK",
]
flights[export_columns].to_csv(all_flights_path, index=False)


C:\Users\Aditi Chauhan\AppData\Local\Temp\ipykernel_18420\1836426976.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  flights = pd.concat(frames, ignore_index=True)


## 3. Build consecutive-aircraft pairs

A pair is kept only when two sequential events share the same tail and airport handoff, both flights are operated (not cancelled/diverted), the previous arrival and current departure/arrival values exist, and scheduled plus actual turnaround intervals are between 0 and 24 hours. These rules avoid inventing continuity across implausible gaps. Disruption events with reported departure times remain in the event stream as barriers.

The pair table includes observed current-flight delay fields for analysis. For prediction, do **not** feed current-flight `DEP_DELAY_MIN`, actual times, or other post-outcome fields into the model: they reveal the answer. Use only information available at the chosen prediction time, and fit any imputation/encoding/scaling after the train/test split.

In [5]:
operated = flights.loc[
    flights["CANCELLED"].eq(0)
    & flights["DIVERTED"].eq(0)
    & flights["TAIL_NUM"].notna()
    & flights["ACTUAL_DEP_UTC"].notna()
    & flights["ACTUAL_ARR_UTC"].notna()
    & flights["SCHEDULED_DEP_UTC"].notna()
    & flights["SCHEDULED_ARR_UTC"].notna()
    & flights["DEP_DELAY"].notna()
    & flights["ARR_DELAY_NEW"].notna()
].copy()
# Keep cancelled/diverted events that have a reported actual departure in
# the ordered event stream. They act as barriers, so filtering them out
# before shifting cannot create a false link across a disruption.
sequence_events = flights.loc[
    flights["TAIL_NUM"].notna() & flights["ACTUAL_DEP_UTC"].notna()
].copy()
sequence_events = sequence_events.sort_values(
    ["TAIL_NUM", "ACTUAL_DEP_UTC", "FLIGHT_DATE", "OP_CARRIER_FL_NUM"],
    kind="mergesort",
).reset_index(drop=True)
grouped = sequence_events.groupby("TAIL_NUM", sort=False)
sequence_events["PREV_TAIL_NUM"] = grouped["TAIL_NUM"].shift(1)
for column, output in [
    ("DEST", "PREV_DEST"),
    ("FLIGHT_DATE", "PREV_FLIGHT_DATE"),
    ("OP_CARRIER_FL_NUM", "PREV_FLIGHT_NUM"),
    ("ORIGIN", "PREV_ORIGIN"),
    ("SCHEDULED_ARR_UTC", "PREV_SCHEDULED_ARR_UTC"),
    ("ACTUAL_ARR_UTC", "PREV_ACTUAL_ARR_UTC"),
    ("ARR_DELAY_NEW", "PREV_ARR_DELAY_MIN"),
    ("CANCELLED", "PREV_CANCELLED"),
    ("DIVERTED", "PREV_DIVERTED"),
]:
    sequence_events[output] = grouped[column].shift(1)

sequence_events["SCHEDULED_TURNAROUND_SLACK_MIN"] = (
    sequence_events["SCHEDULED_DEP_UTC"] - sequence_events["PREV_SCHEDULED_ARR_UTC"]
).dt.total_seconds() / 60
sequence_events["ACTUAL_TURNAROUND_MIN"] = (
    sequence_events["ACTUAL_DEP_UTC"] - sequence_events["PREV_ACTUAL_ARR_UTC"]
).dt.total_seconds() / 60
valid_pairs = (
    sequence_events["CANCELLED"].eq(0)
    & sequence_events["DIVERTED"].eq(0)
    & sequence_events["DEP_DELAY"].notna()
    & sequence_events["ACTUAL_ARR_UTC"].notna()
    & sequence_events["SCHEDULED_ARR_UTC"].notna()
    & sequence_events["PREV_TAIL_NUM"].notna()
    & sequence_events["PREV_CANCELLED"].eq(0)
    & sequence_events["PREV_DIVERTED"].eq(0)
    & sequence_events["PREV_ARR_DELAY_MIN"].notna()
    & sequence_events["PREV_ACTUAL_ARR_UTC"].notna()
    & sequence_events["PREV_SCHEDULED_ARR_UTC"].notna()
    & sequence_events["PREV_DEST"].eq(sequence_events["ORIGIN"])
    & sequence_events["SCHEDULED_TURNAROUND_SLACK_MIN"].between(0, MAX_TURNAROUND_HOURS * 60)
    & sequence_events["ACTUAL_TURNAROUND_MIN"].between(0, MAX_TURNAROUND_HOURS * 60)
)
pairs = sequence_events.loc[valid_pairs].copy()
pairs["DEP_DEL15"] = (pairs["DEP_DELAY"] >= 15).astype("int8")
pairs = pairs.rename(
    columns={
        "FLIGHT_DATE": "FLIGHT_DATE",
        "OP_UNIQUE_CARRIER": "CARRIER",
        "OP_CARRIER_FL_NUM": "FLIGHT_NUM",
        "TAIL_NUM": "TAIL_NUM",
        "ORIGIN": "ORIGIN",
        "DEST": "DEST",
        "SCHEDULED_DEP_UTC": "SCHEDULED_DEP_UTC",
        "ACTUAL_DEP_UTC": "ACTUAL_DEP_UTC",
        "DEP_DELAY": "DEP_DELAY_MIN",
        "DEPARTURE_HOUR_LOCAL": "DEPARTURE_HOUR_LOCAL",
        "DAY_OF_WEEK": "DAY_OF_WEEK",
        "MONTH": "MONTH",
    }
)
pair_columns = [
    "PREV_FLIGHT_DATE", "PREV_FLIGHT_NUM", "PREV_ORIGIN", "PREV_DEST", "TAIL_NUM",
    "PREV_SCHEDULED_ARR_UTC", "PREV_ACTUAL_ARR_UTC", "PREV_ARR_DELAY_MIN",
    "FLIGHT_DATE", "CARRIER", "FLIGHT_NUM", "ORIGIN", "DEST",
    "SCHEDULED_DEP_UTC", "ACTUAL_DEP_UTC", "DEP_DELAY_MIN", "DEP_DEL15",
    "SCHEDULED_TURNAROUND_SLACK_MIN", "ACTUAL_TURNAROUND_MIN",
    "DEPARTURE_HOUR_LOCAL", "DAY_OF_WEEK", "MONTH",
]
pair_path = OUTPUT_DIR / f"wn_aircraft_pairs_{MONTH_LABEL}.csv"
pairs[pair_columns].to_csv(pair_path, index=False)


## 4. Summarize cancellations

Cancellation reason codes are grouped by month and labeled using the BTS code definitions. Missing reasons are preserved as `Unknown / blank`; they are not imputed because the reason is not known.

In [6]:
cancellation_code_labels = {
    "A": "Air carrier",
    "B": "Weather",
    "C": "National aviation system",
    "D": "Security",
}
cancelled = flights.loc[flights["CANCELLED"].eq(1), ["YEAR", "MONTH", "CANCELLATION_CODE"]].copy()
cancelled["CANCELLATION_CODE"] = cancelled["CANCELLATION_CODE"].fillna("").astype(str).str.strip()
cancelled["CANCELLATION_CODE"] = cancelled["CANCELLATION_CODE"].replace("", "Unknown / blank")
cancellation_summary = (
    cancelled.groupby(["YEAR", "MONTH", "CANCELLATION_CODE"], dropna=False)
    .size()
    .rename("CANCELLED_FLIGHTS")
    .reset_index()
)
cancellation_summary["CODE_DESCRIPTION"] = cancellation_summary["CANCELLATION_CODE"].map(
    cancellation_code_labels
).fillna("Unknown / not in BTS A-D list")
cancellation_summary_path = REPORT_DIR / "day2_cancellation_summary.csv"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
cancellation_summary.to_csv(cancellation_summary_path, index=False)


## 5. Write data quality and profile outputs

The outputs report how many flights and pairs remain, missing or unresolved timestamps, class balance, and cancellation counts. Review these before modeling; do not treat missing actual times for cancellations as ordinary values to fill.

In [7]:
pair_tail_counts = pairs.groupby("TAIL_NUM").size()
summary = {
    "carrier": CARRIER,
    "period": MONTH_LABEL,
    "raw_files": file_summary,
    "wn_flight_rows": int(len(flights)),
    "wn_cancelled_rows": int(flights["CANCELLED"].sum()),
    "wn_diverted_rows": int(flights["DIVERTED"].sum()),
    "wn_eligible_operated_flights_with_required_times_and_delays": int(len(operated)),
    "pairs_linked_by_tail_and_matching_airport": int(len(pairs)),
    "distinct_tails_in_pairs": int(pair_tail_counts.size),
    "tail_groups_with_two_or_more_pairs": int((pair_tail_counts >= 2).sum()),
    "pair_target_rate_dep_del15": float(pairs["DEP_DEL15"].mean()) if len(pairs) else None,
    "pair_rows_without_prev_arrival_delay": int(pairs["PREV_ARR_DELAY_MIN"].isna().sum()),
    "cancellations_by_month_and_code": cancellation_summary.to_dict(orient="records"),
    "airport_timezone_mappings_used": int(len(timezone_map)),
    "missing_or_unresolved_timestamps_in_flight_file": {
        column: int(flights[column].isna().sum())
        for column in ["SCHEDULED_DEP_UTC", "ACTUAL_DEP_UTC", "SCHEDULED_ARR_UTC", "ACTUAL_ARR_UTC"]
    },
    "outputs": [
        str(raw_profile_path.relative_to(ROOT)),
        str(raw_statistics_path.relative_to(ROOT)),
        str(raw_profile_summary_path.relative_to(ROOT)),
        str(all_flights_path.relative_to(ROOT)),
        str(pair_path.relative_to(ROOT)),
        str(cancellation_summary_path.relative_to(ROOT)),
    ],
}
REPORT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = REPORT_DIR / "day2_data_quality.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")


4268

## 6. Inspect sample aircraft chains

Ten deterministic examples make it possible to check route handoffs and turnaround intervals manually. This is a targeted data-quality review, not proof that every linkage is correct.

In [8]:
# Deterministic examples make it easy to inspect route continuity and timing.
sample_tails = sorted(pair_tail_counts.index.astype(str))
if len(sample_tails) > 10:
    positions = [round(i * (len(sample_tails) - 1) / 9) for i in range(10)]
    sample_tails = [sample_tails[i] for i in positions]
sample_path = ANALYSIS_DIR / "day2_sample_chains.md"
lines = [
    "# Sample aircraft chains",
    "",
    "Ten deterministic examples from the linked WN flight pairs. Times below are UTC; turnaround slack is between the previous scheduled arrival and this flight's scheduled departure.",
    "",
]
for tail in sample_tails:
    subset = pairs.loc[pairs["TAIL_NUM"].eq(tail)].head(4)
    lines.extend([f"## Tail `{tail}`", "", "| Previous flight | Current flight | Scheduled slack (min) | Actual turn (min) | Previous arrival delay (min) | Current departure delay (min) |", "|---|---|---:|---:|---:|---:|"])
    for row in subset.itertuples(index=False):
        previous_flight_num = str(int(row.PREV_FLIGHT_NUM)) if pd.notna(row.PREV_FLIGHT_NUM) else "?"
        current_flight_num = str(int(row.FLIGHT_NUM)) if pd.notna(row.FLIGHT_NUM) else "?"
        lines.append(
            f"| {row.PREV_FLIGHT_DATE:%Y-%m-%d} {previous_flight_num} {row.PREV_ORIGIN}->{row.PREV_DEST} | "
            f"{row.FLIGHT_DATE:%Y-%m-%d} {current_flight_num} {row.ORIGIN}->{row.DEST} | "
            f"{row.SCHEDULED_TURNAROUND_SLACK_MIN:.0f} | {row.ACTUAL_TURNAROUND_MIN:.0f} | "
            f"{row.PREV_ARR_DELAY_MIN:.0f} | {row.DEP_DELAY_MIN:.0f} |"
        )
    lines.append("")
sample_path.write_text("\n".join(lines), encoding="utf-8")


5141

## 7. Review the preprocessing run summary

In [9]:
print(json.dumps(summary, indent=2))
print(f"Sample chains: {sample_path.relative_to(ROOT)}")


{
  "carrier": "WN",
  "period": "2026-01_to_2026-04",
  "raw_files": [
    {
      "file": "T_ONTIME_REPORTING_20261027_145650\\T_ONTIME_REPORTING.csv",
      "rows_for_WN": 104002,
      "cancelled_for_WN": 2730,
      "diverted_for_WN": 152,
      "missing_tail_for_WN": 512,
      "unparsed_actual_departures": 2697,
      "unparsed_actual_arrivals": 2752
    },
    {
      "file": "T_ONTIME_REPORTING_20261027_145800\\T_ONTIME_REPORTING.csv",
      "rows_for_WN": 100783,
      "cancelled_for_WN": 1018,
      "diverted_for_WN": 187,
      "missing_tail_for_WN": 122,
      "unparsed_actual_departures": 1003,
      "unparsed_actual_arrivals": 1043
    },
    {
      "file": "T_ONTIME_REPORTING_20261027_150025\\T_ONTIME_REPORTING.csv",
      "rows_for_WN": 122298,
      "cancelled_for_WN": 1911,
      "diverted_for_WN": 285,
      "missing_tail_for_WN": 92,
      "unparsed_actual_departures": 1867,
      "unparsed_actual_arrivals": 1953
    },
    {
      "file": "T_ONTIME_REPORTING_2026